In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from src.data_loader import create_dataloaders, get_class_mapping
from src.models import BaselineCNN
from src.train import train_model, run_grid_search
from src.evaluate import evaluate_model
from src.config import *
from src.plot_utils import AccLossPlot, TrainLossPlot

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_device()
print(f"Hardware configuration: Using device '{device}'\n")

class_mapping = get_class_mapping(RAW_DATA_DIR)
num_classes = len(class_mapping)
print(f"{num_classes} classes found: {class_mapping}\n")

Hardware configuration: Using device 'mps'

3 classes found: {'paper': 0, 'rock': 1, 'scissors': 2}



In [2]:
# Define hyperparameter grid for Stratified K-Fold CV using run_grid_search
param_grid = {
    'lr': [1e-2, 1e-3, 3e-4],
    'batch_size': [16, 32],
    'dropout_rate': [0.3, 0.5],
    'weight_decay': [0, 1e-4],
    'epochs': [20]
}

print('Initiating Stratified K-Fold Grid Search (this may take a while)...')
tuning_results = run_grid_search(BaselineCNN, param_grid, None, RAW_DATA_DIR, device=device, num_classes=num_classes, n_splits=3, seed=RANDOM_SEED, patience=3, min_delta=1e-4)
tuning_results.to_csv('saved_models/grid_search_results.csv', index=False)
print('Grid search complete. Top results:')
print(tuning_results.head())

Initiating Stratified K-Fold Grid Search (this may take a while)...
Starting Grid Search with 24 configurations (Stratified 3-Fold CV)...


Experiment 1/24: {'lr': 0.01, 'batch_size': 16, 'dropout_rate': 0.3, 'weight_decay': 0, 'epochs': 20}
  Fold 1/3 (train 1458 / val 730)

--- Epoch 1/20 ---
[TRAIN Batch 000/092]	Time 1.122s (1.122s)	Loss 1.1326 (1.1326)	Prec@1  18.8 ( 18.8)
[TRAIN Batch 010/092]	Time 0.081s (0.175s)	Loss 1.1032 (1.1117)	Prec@1  43.8 ( 33.5)
[TRAIN Batch 020/092]	Time 0.081s (0.133s)	Loss 1.1063 (1.1071)	Prec@1  25.0 ( 31.8)
[TRAIN Batch 030/092]	Time 0.082s (0.119s)	Loss 1.0962 (1.0982)	Prec@1  43.8 ( 34.5)
[TRAIN Batch 040/092]	Time 0.089s (0.111s)	Loss 1.1233 (1.0965)	Prec@1  37.5 ( 36.0)
[TRAIN Batch 050/092]	Time 0.080s (0.105s)	Loss 1.0911 (1.0928)	Prec@1  31.2 ( 37.3)
[TRAIN Batch 060/092]	Time 0.081s (0.102s)	Loss 1.0944 (1.0873)	Prec@1  31.2 ( 39.2)
[TRAIN Batch 070/092]	Time 0.080s (0.099s)	Loss 1.0347 (1.0833)	Prec@1  56.2 ( 39.9)
[TRAIN Batch 080/092]	Ti

KeyboardInterrupt: 

In [2]:
# Extract best configuration from tuning results
#best_config = tuning_results.iloc[0]
best_config = {
    'lr': 1e-3,
    'batch_size': 16,
    'dropout_rate': 0.5,
    'weight_decay': 1e-4,
    'epochs': 20
}

best_lr = float(best_config['lr'])
best_batch_size = int(best_config['batch_size'])
best_dropout = float(best_config.get('dropout_rate', 0.0))
best_weight_decay = float(best_config.get('weight_decay', 0.0))
best_epochs = int(best_config.get('epochs', 5))
final_epochs = 20  # Train longer for the final run
print(f"Selected best config: lr={best_lr}, batch_size={best_batch_size}, dropout={best_dropout}, weight_decay={best_weight_decay}, epochs={best_epochs}")

Selected best config: lr=0.001, batch_size=16, dropout=0.5, weight_decay=0.0001, epochs=20


In [ ]:
print(f"\nStarting final training with best parameters: LR={best_lr}, Batch={best_batch_size}, Dropout={best_dropout}")

train_loader, val_loader, test_loader, _ = create_dataloaders(RAW_DATA_DIR, batch_size=best_batch_size)

model = BaselineCNN(num_classes=num_classes, input_shape=(3, IMG_HEIGHT, IMG_WIDTH)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

# %matplotlib notebook
# plotter = AccLossPlot()

model, history = train_model(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    criterion=criterion, 
    optimizer=optimizer, 
    device=device,
    num_epochs=final_epochs,
    patience=5,
    min_delta=1e-4
    # plotter=plotter
)

print("\nEvaluating on the test set...")
test_loss, all_preds, all_labels = evaluate_model(
    model=model, 
    test_loader=test_loader, 
    criterion=criterion, 
    device=device,
    class_mapping=class_mapping,
)

save_path = os.path.join(MODELS_DIR, "convnet_best.pth")
torch.save(model.state_dict(), save_path)
print(f"\nModel weights successfully saved to {save_path}")


Starting final training with best parameters: LR=0.001, Batch=16, Dropout=0.5


<IPython.core.display.Javascript object>


--- Epoch 1/20 ---
[TRAIN Batch 000/096]	Time 0.281s (0.281s)	Loss 1.1183 (1.1183)	Prec@1  25.0 ( 25.0)
[TRAIN Batch 010/096]	Time 0.090s (0.104s)	Loss 1.1185 (1.1006)	Prec@1  18.8 ( 34.1)
[TRAIN Batch 020/096]	Time 0.088s (0.097s)	Loss 1.1103 (1.1005)	Prec@1  25.0 ( 33.0)
[TRAIN Batch 030/096]	Time 0.095s (0.096s)	Loss 1.1167 (1.0998)	Prec@1  18.8 ( 34.3)
[TRAIN Batch 040/096]	Time 0.085s (0.094s)	Loss 1.0964 (1.0994)	Prec@1  37.5 ( 34.0)
[TRAIN Batch 050/096]	Time 0.084s (0.093s)	Loss 1.0922 (1.0989)	Prec@1  37.5 ( 34.6)
[TRAIN Batch 060/096]	Time 0.098s (0.094s)	Loss 1.0993 (1.0986)	Prec@1  31.2 ( 34.8)
[TRAIN Batch 070/096]	Time 0.271s (0.096s)	Loss 1.1053 (1.0991)	Prec@1   6.2 ( 33.7)
[TRAIN Batch 080/096]	Time 0.085s (0.096s)	Loss 1.0933 (1.0989)	Prec@1  43.8 ( 34.0)
[TRAIN Batch 090/096]	Time 0.087s (0.095s)	Loss 1.0967 (1.0989)	Prec@1  37.5 ( 34.4)

===============> Total time 9s	Avg loss 1.0987	Avg Prec@1 34.68 %

[EVAL Batch 000/021]	Time 0.057s (0.057s)	Loss 1.0976 (1.0976)

<IPython.core.display.Javascript object>


Model weights successfully saved to /Users/gabriele/repos/rps-cnn/saved_models/convnet_best.pth
